### `etd_rk43.ipynb`
*Created: September 25, 2026* <br/>
This notebook implements ETD-RK4 with adaptive step sizing, using a 4(3) embedded ETD-RK pair. 

In [ ]:
using NBInclude, LinearAlgebra, Printf
@nbinclude("etd_rk_core.ipynb")
@nbinclude("../../../phi_functions/phi_functions.ipynb")

In [ ]:
function prepare_step!(cache::ETDRK43Cache, state::SolverState, prob::ODEProblem)    

    #Compute all quantities that appear in the formula for computing the next candidate 
   
    @unpack M, N, E, G, P, Q, K, F, B = cache 
    @unpack f, p = prob 
    @unpack t, u, dt = state

    #Extract the linear and nonlinear terms of the SplitFunction
    A = f.f1.f.A 
    f = prob.f.f2
    
    #Compute phi matrices 
    M .= dt*A
    N .= 0.5*dt*A

    E .= phi(M, 0)
    G .= phi(N, 0)

    P[1] .= phi(M, 1)
    P[2] .= phi(M, 2)
    P[3] .= phi(M, 3)

    Q[1] .= phi(N, 1)
    Q[2] .= phi(N, 2)
    Q[3] .= phi(N, 3)

    #Compute vectors
    K[1] .= u
    F[1] .= f(K[1], p, t)
    
    K[2] .= G*u + (dt/2)*Q[1]*F[1]
    F[2] .= f(K[2], p, t + dt/2)

    K[3] .= G*u + (dt/2)*((Q[1] - Q[2])*F[1] + Q[2]*F[2])
    F[3] .= f(K[3], p, t + dt/2)

    K[4] .= E*u + dt*((P[1] - 2*P[2])*F[1] + 2*P[2]*F[3]) 
    F[4] .= f(K[4], p, t + dt) 

    #Compute B matrices
    B[1] .= P[1] - 3*P[2] + 4*P[3]
    B[2] .=        2*P[2] - 4*P[3]
    B[3] .=        2*P[2] - 4*P[3]
    B[4] .=         -P[2] + 4*P[3]
end 

In [ ]:
function compute_candidate!(cache::ETDRK43Cache, state::SolverState, prob::ODEProblem)

    @unpack E, B, F = cache 
    @unpack dt, u, t = state 

    state.u_candidate .= E*u + dt*(B[1]*F[1] + B[2]*F[2] + B[3]*F[3] + B[4]*F[4])
end 

In [ ]:
function compute_error_estimate!(cache::ETDRK43Cache, state::SolverState, prob::ODEProblem)
    
    @unpack B, F = cache
    @unpack u_candidate, dt, t = state
    p = prob.p
    f = prob.f.f2
      
    state.error_estimate = dt*B[4]*(f(u_candidate, p, t + dt)  - F[4])
end 

In [ ]:
error_estimate_order(::ETDRK43) = 4 
error_estimate_order(::ETDRK53) = 4

function compute_error_scale(alg::ExponentialRKAlgorithm, tol::Float64, error_estimate::U, safety::Float64) where {U}
    q = error_estimate_order(alg)
    return safety * (tol / norm(error_estimate))^(1/q)
end

function dt_scale_factor(alg::ETDRK43, s::Float64, adaptive_options::AdaptiveOptions)
     
    #This function implements the step update method of Whalen (2015)
    @unpack μ_min, μ_c, μ_e, μ_max = adaptive_options

    μ = 1.0 
    
    if s ≤ μ_min
        μ = μ_min
    elseif μ_min ≤ s < μ_c
        μ = s 
    elseif μ_c ≤ s < 1.0
        μ = μ_c 
    elseif 1.0 ≤ s < μ_e 
        μ = 1.0
    elseif μ_e ≤ s ≤ μ_max
        μ = s 
    elseif s > μ_max
        μ = μ_max
    end 

    return μ
end 

In [ ]:
function process_step!(alg::ETDRK43, state::SolverState, adaptive_options::AdaptiveOptions)
    """
    Compute `step_result` (:success or :failure) and `dt_scale_factor` for updating the step size. 
    """
    
    @unpack u, u_candidate, error_estimate = state
    @unpack abstol, reltol, safety, dt_min, dt_max, μ_min, μ_max, μ_c, μ_e = adaptive_options

    tol = abstol + reltol * max(norm(u), norm(u_candidate))
    s = compute_error_scale(alg, tol, error_estimate, safety)
    μ = dt_scale_factor(alg, s, adaptive_options)

    step_result = μ < 1 ? :failure : :success 

    return step_result, μ
end 

In [ ]:
function exponential_rk_step!(cache::ExponentialRKCache, state::SolverState, prob::ODEProblem, alg::ExponentialRKAlgorithm, 
                              adaptive_options::AdaptiveOptions)
        
    #1) update algorithm cache (i.e. get everything needed to compute the candidate u)
    prepare_step!(cache, state, prob)   

    #2) Compute candidate u & update adaptive cache (which stores dt, u_current, u_candidate)
    compute_candidate!(cache, state, prob)

    #3) Compute error estimate
    compute_error_estimate!(cache, state, prob)

    #4) Decide whether to accept or reject the step, and update dt accordingly 
    step_result, μ = process_step!(alg, state, adaptive_options)
   
    #6) Update the state & record new `t` and `u` if step was successful
    if step_result == :success
        state.naccept += 1 
        state.t += state.dt           #Increment `t`
        state.u .= state.u_candidate   #*NOTE: Must use ".=" (NOT "=") to avoid aliasing error!!!!

        #Record the accepted `t` and `u`
        push!(state.saved_t, state.t)
        push!(state.saved_u, copy(state.u))
    else
        state.nreject += 1 
    end

    #5) Update step size & step counter 
    state.dt = μ * state.dt 
    state.current_step += 1 
    return nothing
end 

In [ ]:
function get_initial_dt(tspan; target_steps::Int = 100, dt_min::Float64 = eps(Float64), dt_max::Float64 = 100.0)

    t0, tf = tspan

    t0 < tf || throw(ArgumentError("`tspan` must satisfy `tspan[1]` < `tspan[2]`."))
    target_steps > 0 || throw(ArgumentError("`target_steps` must be positive."))
    0 < dt_min < dt_max || throw(ArgumentError("`dt_min` and `dt_max` must satisfy 0 < `dt_min` < `dt_max`."))
    
    h = (tf - t0) / target_steps
    return clamp(h, dt_min, dt_max)
end 

In [ ]:
function solve(prob::ODEProblem, alg::ExponentialRKAlgorithm; saveat::Vector{Float64} = Float64[], 
               maxiters::Int = 100_000, dt = nothing, abstol = 1e-6, reltol = 1e-3, 
               safety = 0.9, dt_min = 1e-10, dt_max = 1.0, μ_min = 0.4, μ_max = 2.5, μ_c = 0.85, μ_e = 1.25)  

    """
    Solve the semilinear ODE u_t = Au + f(u,p,t), u(t0) = u0 on the time interval [t0, tf]. 
    
    PARAMTERS
    ---------
    prob :: an ODEProblem instance 
    alg :: ETDRK43() or ETDRK53() [only options for now] 
    saveat :: times at which to save the solution
    maxiters :: maximum number of step attempts  
    dt :: initial dt value (will change due to adaptive time stepping)
    abstol :: absolute tolerance for the adaptive time stepping
    reltol :: relative tolerance for the adaptive time stepping
    dt_min :: minimum allowed step size
    dt_max :: maximum allowed step size 
    safety :: adaptive time stepping parameter
    μ_min, μ_max, μ_c, μ_e :: time stepping parameters specific to the ETDRK43() and ETDRK53() methods. 
        
    RETURNS
    -------
    sol.u :: solution iterates at the recorded time values
    sol.t :: times at which the solution was recorded
    prob :: the ODE problem that was passed to the solver
    state :: the final state of the solver
    """
        
    #Note: the parameters in the last line are specific to *my implementation*. The others are all part of OrdinaryDiffEq.jl's `solve()` function. 
            
    #INPUT VALIDATION
    prob.f isa SplitFunction || throw(ArgumentError("$(alg) requires a problem constructed with `SplitODEProblem()`."))
    maxiters > 0 || throw(ArgumentError("`maxiters` must be a positive integer."))
    #Check that dt > 0 (if supplied)
    #Check that all additional parameters are positive

            
    #Unpack the necessary ingredients
    @unpack f, u0, tspan, p = prob 

    #Extract the constant matrix
    A = f.f1.f.A                        

    #Convert t0, tf to float's (Note: `float()` preserves the existing precision, whereas Float64 does not) 
    t0, tf = float.(tspan)
    u0 = float.(u0)

    #Create array to store the saved time values
    t = [t0]          
    sizehint!(t, maxiters)

    #Ensure u0 is a float or vector of floats
    u = [u0]
    sizehint!(u, maxiters)

    #Construct AdaptiveOptions instance 
    adaptive_options = AdaptiveOptions(abstol = abstol, reltol = reltol, safety = safety, dt_min = dt_min, dt_max = dt_max,
                                       μ_min = μ_min, μ_max = μ_max, μ_c = μ_c, μ_e = μ_e)

    #Set the initial time step
    dt = isnothing(dt) ? get_initial_dt(tspan; target_steps = 100, dt_min = dt_min, dt_max = dt_max) : copy(dt)

    #Initialize cache and state
    cache = initialize_cache(u0, alg)
    state = initialize_state(t0, u0, dt)

    ##### START THE LOOP ###########
    
    println("Starting integration loop...")

    while state.t < tf && state.current_step ≤ maxiters
        exponential_rk_step!(cache, state, prob, alg, adaptive_options)
    end 

    @printf("Step acceptance rate: %.2f %%", (state.naccept / (state.current_step - 1)) * 100)
   
     #return (cache = cache, state = state, options = options)
    return (u = state.saved_u, t = state.saved_t, prob = prob, state = state)
end